# CUDA GEMM Optimization: Phase-by-Phase Implementation

This notebook walks through a progression of CUDA GEMM kernels, from naive to tensor-core optimized.

**Target GPU:** NVIDIA T4 (Colab default) — Compute Capability 7.5
**Problem size:** M=N=K=4096

Each phase builds on the previous one. **Run cells in order.**

## Setup

First, upload the project files or clone from git. Then verify CUDA is available.

In [ ]:
import subprocess, sys, os, glob, shutil, time, textwrap

# Check CUDA
!nvcc --version
!nvidia-smi

# Print device info
!cat /proc/driver/nvidia/gpus/0/information 2>/dev/null || echo "No GPU info file"

In [ ]:
from google.colab import files
import zipfile, io

print("Upload the cuda-gemm project zip file:")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(uploaded[filename]), 'r') as zip_ref:
            zip_ref.extractall('/content/')
        print(f"Extracted {filename}")

!ls -la /content/cuda-gemm/ 2>/dev/null || !ls -la /content/GEMM/ 2>/dev/null

# Find and set working directory
if os.path.isdir('/content/cuda-gemm'):
    os.chdir('/content/cuda-gemm')
elif os.path.isdir('/content/GEMM'):
    os.chdir('/content/GEMM')
print(f"Working directory: {os.getcwd()}")

### Helper Functions

We'll define a reusable compile-and-run helper.

In [ ]:
def compile_and_run(kernel_name, extra_flags=""):
    src = f"kernels/{kernel_name}.cu"
    exe = f"kernels/{kernel_name}"
    arch = "-arch=sm_75"  # T4
    cmd = f"nvcc {arch} -std=c++17 -O3 {extra_flags} -o {exe} {src} -lcublas -lcudart"
    print(f"Compiling: {cmd}")
    ret = os.system(cmd)
    if ret != 0:
        print("COMPILATION FAILED")
        return False
    print(f"Running {kernel_name}...")
    ret = os.system(f"./{exe}")
    if ret != 0:
        print("EXECUTION FAILED or VERIFICATION FAILED")
        return False
    print("PASS")
    return True

results = {}

---
## Phase 0: Naive Kernel (Baseline)

Each thread computes one output element. Every thread re-reads the entire row of A and column of B from global memory. This is the baseline — expect ~1-3% of cuBLAS performance.

In [ ]:
print("=" * 60)
print("Phase 0: Naive Kernel")
print("=" * 60)
ok = compile_and_run("00_naive")
results["00_naive"] = ok

---
## Phase 1: Global Memory Coalescing

No separate kernel — this is a note about checking access patterns. The naive kernel already has coalesced B reads (`B[k*N + col]` where `col = threadIdx.x`), which is the most performance-critical access. If you were adapting this for a transposed layout, you'd swap the thread-to-element mapping.

**Key insight:** adjacent threads (consecutive `threadIdx.x`) should read consecutive global memory addresses. Our naive kernel's `col` maps to `threadIdx.x`, so `B[k*N + col]` is coalesced. The A read `A[row*K + k]` varies by `threadIdx.y` so it's not coalesced, but since A is accessed in the inner loop across `k`, this is the less critical path.

**Result:** The naive kernel above is already correctly configured for coalesced B access. Moving on.

---
## Phase 2: Shared-Memory Tiling (Block Tiling)

Load `BLOCK_SIZE x BLOCK_SIZE` tiles of A and B into shared memory once, then reuse across all threads in the block. This reduces global memory traffic from `O(K)` per thread to `O(K/BLOCK_SIZE)` per thread.

In [ ]:
print("=" * 60)
print("Phase 2: Shared-Memory Tiling")
print("=" * 60)
ok = compile_and_run("02_shared_tiled")
results["02_shared_tiled"] = ok

---
## Phase 3: 1D Register Blocking (Thread Coarsening)

Each thread computes `TM` output elements (down a column) instead of one. This amortizes each shared-memory load across more FMAs. Shared memory bandwidth is the new bottleneck after Phase 2, and this is the single biggest speedup in the progression.

In [ ]:
print("=" * 60)
print("Phase 3: 1D Register Blocking")
print("=" * 60)
ok = compile_and_run("03_1d_blocktiling")
results["03_1d_blocktiling"] = ok

---
## Phase 4: 2D Register Blocking (Micro-Tiling)

Each thread computes a `TM x TN` micro-tile of the output using outer-product-style updates. `TM + TN` shared-memory reads produce `TM * TN` FMAs, pushing arithmetic intensity further. This is close to what cuBLAS/CUTLASS do internally.

In [ ]:
print("=" * 60)
print("Phase 4: 2D Register Blocking")
print("=" * 60)
ok = compile_and_run("04_2d_blocktiling")
results["04_2d_blocktiling"] = ok

---
## Phase 5: Vectorized Memory Access (float4)

Load/store 128 bits at a time using `float4` to reduce instruction count and better saturate memory bandwidth. Applied to global→shared loads and shared→register loads.

In [ ]:
print("=" * 60)
print("Phase 5: Vectorized Memory Access")
print("=" * 60)
ok = compile_and_run("05_vectorized")
results["05_vectorized"] = ok

---
## Phase 6: Bank-Conflict-Free Shared Memory

Shared memory has 32 banks. The `As[threadRow*TM + m][k]` access pattern in Phase 5 can cause bank conflicts. Fix: pad `As` to `[BM][BK + 1]` so consecutive rows don't map to the same bank.

**Recommended:** Profile with Nsight Compute to measure `l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld` before and after.

In [ ]:
print("=" * 60)
print("Phase 6: Bank-Conflict-Free Shared Memory")
print("=" * 60)
ok = compile_and_run("06_bank_conflict_free")
results["06_bank_conflict_free"] = ok

---
## Phase 7: Double Buffering / Software Pipelining

Use two shared-memory buffers. While computing on one buffer, prefetch the next tile into the other. The compute phase hides the latency of the next global-memory load.

In [ ]:
print("=" * 60)
print("Phase 7: Double Buffering")
print("=" * 60)
ok = compile_and_run("07_double_buffered")
results["07_double_buffered"] = ok

---
## Phase 8: Fully Vectorized Memory Access (float4 A + B + C)

Extend float4 vectorization to **all** memory operations: A global→shared loads, B global→shared loads, B shared→register loads, and C register→global stores. Previously only B was vectorized. Also uses padding on As to avoid bank conflicts.

In [ ]:
print("=" * 60)
print("Phase 8: Fully Vectorized (float4 A + B + C)")
print("=" * 60)
ok = compile_and_run("08_vectorized_all")
results["08_vectorized_all"] = ok

---
## Phase 9: Tensor Cores via WMMA

Use `nvcuda::wmma` to drive T4's 2nd-gen tensor cores. FP16 input, FP32 accumulator/output. Each `wmma::mma_sync` does a 16x16x16 MMA in one instruction — vastly higher throughput than scalar FMA.

**Relevance to thesis:** This is a warm-up for mixed-precision attention kernel work. Extending to INT8 fragments (`wmma::experimental::precision::s8`) is a natural next step.

In [ ]:
print("=" * 60)
print("Phase 9: Tensor Core WMMA GEMM")
print("=" * 60)
ok = compile_and_run("09_wmma_tensorcore")
results["09_wmma_tensorcore"] = ok

---
## Phase 10: Multi-Tile WMMA Tensor Core

Each warp computes 2x2 WMMA tiles (32x32 output region) instead of one, increasing compute density. Uses uint4 vectorized loads (8 half elements per 128-bit load) for both A and B global->shared transfers.

In [ ]:
print("=" * 60)
print("Phase 10: Multi-Tile WMMA Tensor Core")
print("=" * 60)
ok = compile_and_run("10_wmma_multi")
results["10_wmma_multi"] = ok

---
## Phase 11: Direct-Load WMMA Tensor Core

Loads B directly into column-major layout in shared memory, eliminating the explicit transpose and one __syncthreads() per tile. Uses uint2 vectorized global loads with `__restrict__` and `__launch_bounds__` hints for better compiler optimization.

In [ ]:
print("=" * 60)
print("Phase 11: Direct-Load WMMA Tensor Core")
print("=" * 60)
ok = compile_and_run("11_wmma_direct")
results["11_wmma_direct"] = ok

---
## Phase 12: Optimized WMMA Tensor Core

BM=128 for more data reuse, 512 threads (50% occupancy), explicit named fragment variables (no arrays) for better compiler optimization, uint4 vectorized loads, B transpose in shared memory.

In [ ]:
print("=" * 60)
print("Phase 12: Optimized WMMA Tensor Core")
print("=" * 60)
ok = compile_and_run("12_wmma_opt")
results["12_wmma_opt"] = ok

---
## Combined Benchmark

Run all kernels in a single pass and compare against cuBLAS.

In [ ]:
print("=" * 60)
print("Combined Benchmark: All Kernels vs cuBLAS")
print("=" * 60)

arch = "-arch=sm_75"
kernel_files = " ".join([
    "kernels/00_naive.cu",
    "kernels/02_shared_tiled.cu",
    "kernels/03_1d_blocktiling.cu",
    "kernels/04_2d_blocktiling.cu",
    "kernels/05_vectorized.cu",
    "kernels/06_bank_conflict_free.cu",
    "kernels/07_double_buffered.cu",
    "kernels/08_vectorized_all.cu",
    "kernels/09_wmma_tensorcore.cu",
    "kernels/10_wmma_multi.cu",
    "kernels/11_wmma_direct.cu",
    "kernels/12_wmma_opt.cu",
])

cmd = f"nvcc {arch} -std=c++17 -O3 -DBENCHMARK_ALL -o bench/benchmark_all bench/benchmark_all.cu {kernel_files} -lcublas -lcudart"
print(f"Compiling: nvcc {arch} -std=c++17 -O3 -DBENCHMARK_ALL -o bench/benchmark_all ...")
ret = os.system(cmd)
if ret == 0:
    !./bench/benchmark_all
else:
    print("COMPILATION FAILED")

---
## Optional: Autotuning

Sweep tile-size configurations (BM, BN, BK, TM, TN) to find the best for this GPU.

In [ ]:
print("Running autotuner (this may take a while)...")
!python3 autotune.py

---
## Summary

Fill in the table below with your results from each phase.

In [ ]:
print("Phase Results:")
print("=" * 40)
for name, ok in results.items():
    status = "PASS" if ok else "FAIL"
    print(f"  {name:30s}: {status}")
print("=" * 40)
print("\nBenchmark table (fill from combined benchmark output):")
print('''
| Level | Technique | GFLOPS | % of cuBLAS |
|-------|-----------|--------|-------------|
| 0 | Naive | | |
| 2 | Shared memory tiling | | |
| 3 | 1D register blocking | | |
| 4 | 2D register blocking | | |
| 5 | Vectorized | | |
| 6 | Bank-conflict-free | | |
| 7 | Double buffering | | |
| 8 | Fully vectorized (A+B+C) | | |
| 9 | Tensor core (WMMA) | | |
| 10 | Multi-tile WMMA | | |
| 11 | Direct-load WMMA | | |
| 12 | Optimized WMMA | | |
| — | cuBLAS (reference) | | |
''')